# DES431 — Recommendation System
**Dataset:** MovieLens 1M | **Metric:** Precision@10

## 1. Data Loading & Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import ast
import gc
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from surprise import SVD, Dataset, Reader


# Define expected valid values based on dataset documentation
VALID_AGES = {1, 18, 25, 35, 45, 50, 56}
VALID_OCCUPATIONS = set(range(0, 21))
VALID_GENDERS = {'m', 'f'}
VALID_RATINGS = set(range(1, 6)) #Ratings are made on a 5-star scale
VALID_GENRES = {
    "action", "adventure", "animation", "children's", "comedy", "crime",
    "documentary", "drama", "fantasy", "film-noir", "horror", "musical",
    "mystery", "romance", "sci-fi", "thriller", "war", "western"
}

In [ ]:
# Load dataset (already in the same directory)
train = pd.read_csv("train.csv") # columns: user_id, movie_id, rating, timestamp
user_load = pd.read_csv('users.dat', sep='::', engine='python', header=None, names=['user_id', 'gender', 'age', 'occupation', 'zip-code'])
movie_load = pd.read_csv('movies.dat', sep='::', engine='python', header=None, names=['movie_id', 'title', 'genres'], encoding='latin1')
val   = pd.read_csv("val.csv")            # one row per user, rating > 3 (relevant item)

# Structure Check
print("--- Training set Structure ---")
print(train.info())
print("--- User data Structure ---")
print(user_load.info())
print("--- Movies data Structure ---")
print(movie_load.info())
print("--- Validation dataset Structure ---")
print(val.info())

print()

In [ ]:
# Function to clean and lowercase the metadata
def preprocess_strings(users, movies):
    # Lowercase gender in users.dat
    users['gender'] = users['gender'].str.lower()
    
    # Lowercase title and genres in movie_data.dat
    movies['title'] = movies['title'].str.lower()
    movies['genres'] = movies['genres'].str.lower()
    
    print("Pre-processing complete: title, genres, and gender are now lowercase.")
    return users, movies

# Apply the pre-processing
user_data, movie_data = preprocess_strings(user_load, movie_load)

In [ ]:
# Check for completely identical rows
duplicate_rows = train.duplicated().sum()
duplicate_rows_user = user_data.duplicated().sum()
duplicate_rows_movie = movie_data.duplicated().sum()
print(f"Number of completely identical rows: ")
print(f"training set: {duplicate_rows}")
print(f"User data: {duplicate_rows_user}")
print(f"User data: {duplicate_rows_movie}")

In [ ]:
def validate_datasets(train, users, movies):
    print("=== Starting Data Validation ===\n")
    
    #__________ 1. Validation for User Data __________________
    print("--- Checking User Data ---")
    age_check = users[~users['age'].isin(VALID_AGES)]
    occ_check = users[~users['occupation'].isin(VALID_OCCUPATIONS)]
    gen_check = users[~users['gender'].isin(VALID_GENDERS)]
    uid_range = users[(users['user_id'] < 1) | (users['user_id'] > 6040)]
    
    print(f"- Invalid ages found: {len(age_check)}")
    print(f"- Invalid occupations found: {len(occ_check)}")
    print(f"- Invalid genders found: {len(gen_check)}")
    print(f"- user_ids out of range (1-6040): {len(uid_range)}")
    
    # Check for zip-codes that aren't numeric or the wrong length
    # Standard US Zip is 5 digits, or 10 digits if it's ZIP+4 (55103-1006)
    weird_zip = user_data[~user_data['zip-code'].str.match(r'^\d{5}(-\d{4})?$')]
    print(f"\nNumber of users with non-standard zip-codes: {len(weird_zip)}")

    # Preview the 'weird' ones
    if len(weird_zip) > 0:
        print(weird_zip['zip-code'].head())
        
    # Check for Duplicate user id
    duplicates = users[users.duplicated(subset=['user_id'], keep=False)]
    print(f"\n- Number of user with duplicate id: {len(duplicates)}")

    #__________ 2. Validation for Movie Data ______________
    print("\n --- Checking Movie Data ---")
    mid_range = movies[(movies['movie_id'] < 1) | (movies['movie_id'] > 3952)]
    
    # Check if all genres are from the allowed list
    all_genres = set()
    movies['genres'].str.split('|').apply(all_genres.update)
    invalid_genres = all_genres - VALID_GENRES
    
    print(f"- movie_ids out of range (1-3952): {len(mid_range)}")
    print(f"- Unexpected genres found: {invalid_genres if invalid_genres else 'None'}")

    # Check for Duplicate Movie titles (Hand-entry errors)
    duplicates = movie_data[movie_data.duplicated(subset=['title'], keep=False)]
    print(f"- Number of movies with duplicate titles: {len(duplicates)}")

    # 3. Validation for Training Set
    print("\n --- Checking Training Set ---")
    rat_check = train[~train['rating'].isin(VALID_RATINGS)]
    train_uid_range = train[(train['user_id'] < 1) | (train['user_id'] > 6040)]
    train_mid_range = train[(train['movie_id'] < 1) | (train['movie_id'] > 3952)]
    
    print(f"- Invalid Ratings found (not 1-5): {len(rat_check)}")
    print(f"- user_id in train set out of range: {len(train_uid_range)}")
    print(f"- movie_id in train set out of range: {len(train_mid_range)}")

    # 4. Referential Integrity (Foreign Key Checks)
    print("\n --- Checking Referential Integrity ---")
    # Ensure every user in train exists in users file
    missing_users = train[~train['user_id'].isin(users['user_id'])]['user_id'].nunique()
    # Ensure every movie in train exists in movies file
    missing_movies = train[~train['movie_id'].isin(movies['movie_id'])]['user_id'].nunique()
    
    print(f"- Users in train set missing from user_data: {missing_users}")
    print(f"- Movies in train set missing from movie_data: {missing_movies}")
    
    # Count how many user_id-movie_id pairs are duplicates
    duplicate_count = train.duplicated(subset=['user_id', 'movie_id']).sum()
    print(f"- Number of duplicate user-item interactions: {duplicate_count}")

    print("\n=== Validation Complete ===")

# Execute validation
validate_datasets(train, user_data, movie_data)

## 2. Content-Based Filtering

**Idea:** Build a genre-vector profile per user (rating-weighted average of seen-movie vectors), then rank unseen movies by cosine similarity to that profile.

**Normalization:** `norm_score = (cosine + 1) / 2` → [0, 1]

In [ ]:
# ------------------------------------------------------------
# Step 1: Build item profiles (binary genre vectors)
# ------------------------------------------------------------
all_genres = sorted(
    {g for genres in movie_data["genres"] for g in genres.split("|")}
)

def encode_genres(genre_str, all_genres):
    movie_genres = set(genre_str.split("|"))
    return np.array([1 if g in movie_genres else 0 for g in all_genres], dtype=np.float32)

item_profiles = np.vstack(
    [encode_genres(row["genres"], all_genres) for _, row in movie_data.iterrows()]
)

item_profile_df = pd.DataFrame(
    item_profiles,
    index=movie_data["movie_id"],
    columns=all_genres
)

In [ ]:
# ------------------------------------------------------------
# Step 2: Build user profiles (rating-weighted average)
# ------------------------------------------------------------
valid_movie_ids = set(movie_data["movie_id"])
train_valid = train[train["movie_id"].isin(valid_movie_ids)].copy()

def build_user_profile(user_ratings: pd.DataFrame) -> np.ndarray:

    rated_vecs = item_profile_df.loc[user_ratings["movie_id"]].values
    ratings = user_ratings["rating"].values.reshape(-1, 1)

    weighted_sum = (rated_vecs * ratings).sum(axis=0)
    weight_total = ratings.sum()

    if weight_total == 0:
        return np.zeros(len(all_genres), dtype=np.float32)

    return (weighted_sum / weight_total).astype(np.float32)

user_profiles = (
    train_valid
    .groupby("user_id")
    .apply(build_user_profile, include_groups=False)
)

user_ids = user_profiles.index.tolist()
user_profile_mat = np.vstack(user_profiles.values)
user_id_to_row = {uid: i for i, uid in enumerate(user_ids)}

print("Item profile matrix shape:", item_profile_df.shape)
print("User profile matrix shape:", user_profile_mat.shape)

In [ ]:
#  Prepare similarity scores and seen/unseen lookup
# ============================================================

# Full similarity matrix: (n_users x n_movies)
sim_matrix = cosine_similarity(user_profile_mat, item_profiles)

movie_ids_list = movie_data["movie_id"].tolist()
all_movie_ids_set = set(movie_ids_list)

# Movies already seen in train.csv
seen_movies = (
    train_valid.groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()
)

print("Similarity matrix shape:", sim_matrix.shape)
print("Total movies:", len(movie_ids_list))

Return a dictionary of all unseen movie scores for one user:
        {movie_id: norm_score}

    For this content-based model:
    - cosine similarity is used as the raw score
    - cosine similarity is mapped from [-1, 1] to [0, 1]
    - scores are clipped into [0, 1]
    

In [ ]:
#  get_all_unseen_scores_content(user_id)
# Returns {movie_id: norm_score} for every unseen movie
# ============================================================

def get_all_unseen_scores_content(user_id: int) -> dict:

    if user_id not in user_id_to_row:
        return {}

    row = user_id_to_row[user_id]
    raw_scores = sim_matrix[row]   # one score per movie
    seen = seen_movies.get(user_id, set())

    unseen_score_dict = {}

    for idx, movie_id in enumerate(movie_ids_list):
        if movie_id in seen:
            continue

        sim_score = raw_scores[idx]

        # Normalize cosine similarity from [-1, 1] -> [0, 1]
        norm_score = (sim_score + 1.0) / 2.0

        # Clip to [0, 1]
        norm_score = float(np.clip(norm_score, 0.0, 1.0))

        unseen_score_dict[movie_id] = norm_score

    return unseen_score_dict

In [ ]:
#  Build the full unseen scoring matrix
# Output columns: user_id, movie_id, norm_score
# ============================================================

def build_unseen_scoring_matrix_content(user_ids_to_score=None) -> pd.DataFrame:
    """
    Build a full unseen scoring matrix with columns:
        user_id, movie_id, norm_score
    """
    if user_ids_to_score is None:
        user_ids_to_score = sorted(train_valid["user_id"].unique())

    rows = []

    for user_id in user_ids_to_score:
        unseen_scores = get_all_unseen_scores_content(user_id)

        for movie_id, norm_score in unseen_scores.items():
            rows.append({
                "user_id": user_id,
                "movie_id": movie_id,
                "norm_score": norm_score
            })

    unseen_df = pd.DataFrame(rows, columns=["user_id", "movie_id", "norm_score"])
    return unseen_df

# Build for all users in training set
content_based_unseen_scores = build_unseen_scoring_matrix_content()

print(content_based_unseen_scores.head())
print(content_based_unseen_scores.shape)

## 3. Global Bias

**Normalization:** `(pred − 1) / 4` → [0, 1]

In [ ]:
# 1. Pre-calculate Global Mean and Biases
mu = train['rating'].mean() 
user_biases = train.groupby('user_id')['rating'].mean() - mu
item_biases = train.groupby('movie_id')['rating'].mean() - mu

# 2. Create a Cartesian Product (All users x All movies)
# Note: This creates all possible combinations.
all_users = val['user_id'].unique()
all_movies = movie_data['movie_id'].unique()

# Create a DataFrame of all combinations
full_grid = pd.MultiIndex.from_product([all_users, all_movies], names=['user_id', 'movie_id']).to_frame(index=False)

# 3. Join with biases
# Map the biases to the grid
full_grid['b_x'] = full_grid['user_id'].map(user_biases).fillna(0)
full_grid['b_y'] = full_grid['movie_id'].map(item_biases).fillna(0)

# 4. Calculate Predicted Rating: mu + bx + by 
full_grid['pred_rating'] = mu + full_grid['b_x'] + full_grid['b_y']

# 5. Normalize to [0, 1]
full_grid['norm_score'] = (full_grid['pred_rating'] - 1.0) / 4.0
full_grid['norm_score'] = full_grid['norm_score'].clip(0.0, 1.0)

# 6. Filter out seen movies (IMPORTANT)
# Get the set of (user_id, movie_id) that are in train
seen_pairs = pd.MultiIndex.from_frame(train[['user_id', 'movie_id']])
mask = pd.MultiIndex.from_frame(full_grid[['user_id', 'movie_id']]).isin(seen_pairs)

# Keep only unseen movies
bias_unseen_scores = full_grid[~mask][['user_id', 'movie_id', 'norm_score']]

print(bias_unseen_scores.head())
print("Final shape:", bias_unseen_scores.shape)


## 6. Latent Factor — SVD

Uses `Surprise` library's SVD (matrix factorization via SGD).

**Normalization:** `(pred − 1) / 4` → [0, 1]

In [ ]:
# 1. Prepare the Data for the Surprise Library
# Define the rating scale (1-5) as per the MovieLens dataset
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(train[['user_id', 'movie_id', 'rating']], reader)
trainset = data.build_full_trainset()

# 2. Initialize and Train the Latent Factor Model (SVD)
# n_factors corresponds to 'l' (Dimension Reduction)
algo = SVD(n_factors=150, 
           n_epochs=10, 
           lr_all=0.01, 
           reg_all=0.1, 
           random_state=42)

# This performs the SGD loop to find the optimal P and Q matrices
algo.fit(trainset)

# 3. Generate Scores for Unseen Movies
# We map movies already seen to avoid recommending them
seen_dict = train.groupby("user_id")["movie_id"].apply(set).to_dict()
all_movie_ids = movie_data["movie_id"].unique()
val_users = val["user_id"].unique() # Focusing on validation users for efficiency

rows = []
for u_id in val_users:
    seen_movies = seen_dict.get(u_id, set())
    for m_id in all_movie_ids:
        if m_id not in seen_movies:
            # Predict the rating using the dot product of latent factors P and Q 
            pred_rating = algo.predict(u_id, m_id).est
            
            # Normalize the score from [1, 5] -> [0, 1] for the ensemble blend
            norm_score = (pred_rating - 1.0) / 4.0
            norm_score = float(np.clip(norm_score, 0.0, 1.0))
            
            rows.append([u_id, m_id, norm_score])

# 4. Create the final Latent Unseen Scores Dataframe
latent_unseen_scores = pd.DataFrame(rows, columns=["user_id", "movie_id", "norm_score"])
print(latent_unseen_scores.head())
print("Shape:", latent_unseen_scores.shape)


## 7. Ensemble

Weighted linear combination of all active models.
Scores are merged on `(user_id, movie_id)` keys (inner join) before blending to prevent row-index misalignment.

Weights are found via **Dirichlet random search** (30 trials, 1,000 sampled users/trial).

In [ ]:
# Example of model_dfs dictionary using your existing variables
model_dataframes = {
    "Content": content_based_unseen_scores,
    "Global": bias_unseen_scores,
    "Latent Factor": latent_unseen_scores
}

In [ ]:
# Finding best weights pair (3 weights with sum = 1 for 3 models)

def optimize_ensemble_fast(model_dfs, val_df, n_trials=50, sample_size=1000):

    sample_users = np.random.choice(val_df['user_id'].unique(), sample_size, replace=False)
    val_sample = val_df[val_df['user_id'].isin(sample_users)]
    relevant_dict = val_sample.set_index('user_id')['movie_id'].to_dict()

    # Data Preparation (Dictionary of Arrays)
    model_scores_arrays = {}
    for name, df in model_dfs.items():
        # Save score in Pivot or Matrix format for easier usage
        model_scores_arrays[name] = df[df['user_id'].isin(sample_users)].sort_values(['user_id', 'movie_id'])

    best_precision = -1
    best_weights = None

    print(f"Starting Random Search for {n_trials} trials...")

    for i in range(n_trials):
        # 3. Randomlizing values with sum = 1.0 (Dirichlet distribution)
        w = np.random.dirichlet(np.ones(len(model_dfs)), size=1)[0]
        weights = {name: w[idx] for idx, name in enumerate(model_dfs.keys())}

        # 4. Calculate Ensemble Score (Numpy Operation)
        first_model = list(model_dfs.keys())[0]
        temp_scores = model_scores_arrays[first_model][['user_id', 'movie_id']].copy()
        temp_scores['norm_score'] = 0.0

        for name, weight in weights.items():
            temp_scores['norm_score'] += model_scores_arrays[name]['norm_score'].values * weight

        # 5. Evaluate
        top_10 = temp_scores.sort_values(['user_id', 'norm_score'], ascending=[True, False]).groupby('user_id').head(10)
        recs = top_10.groupby('user_id')['movie_id'].apply(list).to_dict()
        
        hits = sum(1 for uid, true_mid in relevant_dict.items() if true_mid in recs.get(uid, []))
        precision = (hits * 0.1) / sample_size

        if precision > best_precision:
            best_precision = precision
            best_weights = weights
            print(f"Trial {i+1}: New Best Precision = {best_precision:.4f} with weights {weights}")

    return best_weights


optimal_weights = optimize_ensemble_fast(model_dataframes, val, n_trials=30, sample_size=1000)

In [ ]:
def get_ensemble_scores(weight_dict, model_dfs):
    
    # 1.Start with a base dataframe to merge everything onto
    # We use a subset of one DF to get all user-movie pairs
    ensemble_df = model_dfs['Latent Factor'][['user_id', 'movie_id']].copy()
    
    # # 2. Merge scores from all models into a single DataFrame
    # Each model contributes its normalized score for the same (user_id, movie_id) pairs
    for name in weight_dict.keys():
        temp = model_dfs[name][['user_id', 'movie_id', 'norm_score']]

        # Rename the score column to avoid column name conflicts during merge
        temp = temp.rename(columns={'norm_score': f'score_{name}'})
        
        # Perform inner join to keep only common user–item pairs across all models
        ensemble_df = ensemble_df.merge(temp, on=['user_id', 'movie_id'])
    
    # 3. Compute the final ensemble score using a weighted sum
    # Each model's contribution is scaled by its corresponding weight
    ensemble_df['norm_score'] = sum(
        ensemble_df[f'score_{name}'] * weight_dict[name] 
        for name in weight_dict.keys()
    )
    
    # 4. Return only the required columns for downstream tasks (e.g., ranking)
    return ensemble_df[['user_id', 'movie_id', 'norm_score']] 

final_ensemble_scores = get_ensemble_scores(optimal_weights, model_dataframes)
print(final_ensemble_scores.head())
print("Shape:", final_ensemble_scores.shape)

## 8. Evaluation — Precision@10

**Formula:** Overall Precision@10 = (1/N) Σ Precision@10_i = total_hits / (N × 10)

Val set contains exactly **one** relevant movie per user (rating > 3).

In [ ]:
def process_and_save_recommendations(df_scores, method_name, val_df):
    
    # 1. Ranking: Sort by user and score, then take Top 10 for each user
    # We assume df_scores only contains unseen movies from the previous step
    top_10_df = df_scores.sort_values(['user_id', 'norm_score'], ascending=[True, False])
    top_10_df = top_10_df.groupby('user_id').head(10)
    
    # 2. Formatting: Convert to list objects [m1, m2, ...] per user
    rec_list_df = top_10_df.groupby('user_id')['movie_id'].apply(list).reset_index()
    rec_list_df.columns = ['user_id', 'recommended_movies']
    
    # Save to CSV
    filename = f"recommendations_{method_name.lower().replace(' ', '_')}.csv"
    rec_list_df.to_csv(filename, index=False)
    print(f"File saved: {filename}")
    
    # 3. Evaluation: Precision@10
    # Mapping every user to their target movie_id in the validation set
    relevant_dict = val_df.set_index('user_id')['movie_id'].to_dict()
    
    hits = 0
    total_users = len(relevant_dict)

    print(f"total_users: total_users")
    
    # Convert recommendations to dict for fast lookup (user_id -> list of movie_ids)
    recs_dict = dict(zip(rec_list_df['user_id'], rec_list_df['recommended_movies']))
    
    for user_id, true_movie_id in relevant_dict.items():
        # Get the top 10 list for this user (defaults to empty list if user missing)
        user_recs = recs_dict.get(user_id, [])
        
        # Check if the single relevant movie from val is in the top 10 list
        if true_movie_id in user_recs:
            hits += 1
            
    # Precision@10 is 0.1 for a hit, 0 for a miss
    # Overall Precision@10 = (Total Hits * 0.1) / Total Users
    mean_precision = (hits * 0.1) / total_users if total_users > 0 else 0
    
    print(f"Results for {method_name}:")
    print(f"  Hits: {hits} / {total_users}")
    print(f"  Mean Precision@10: {mean_precision:.4f} ({mean_precision*100:.2f}%)")
    print("-" * 30)
    
    return mean_precision

In [ ]:
# List of dataframes and names to process
all_models = [
    (content_based_unseen_scores, "Content"),
    (bias_unseen_scores, "Global"),
    (latent_unseen_scores,  "Latent Factor"),
    (final_ensemble_scores, "Ensemble_Final")
]

# Process and evaluate each one
for df, name in all_models:
    process_and_save_recommendations(df, name, val)